# 🚂 01 — Dhara: Ingest All Source Data (Bronze Layer)

Ingests all three JSON sources and the AQ CSV into Unity Catalog Bronze Delta tables.

**Sources:**
- `schedules.json` — 389K records: every stop for 5208 trains (arrival/departure times)
- `trains.json` — 5208 trains: type, zone, distance, duration (GeoJSON FeatureCollection)
- `stations.json` — 8990 stations: state, zone, lat/lon (GeoJSON FeatureCollection)
- `india_air_quality.csv` — 3446 rows: PM2.5/NO2 by city/state
- `train_*.csv` — Southern Railway supplementary schedules (wildcard ingestion)

**Databricks depth demonstrated here:** wildcard reads, GeoJSON flattening, CDF on rules table, partitioning.

In [0]:
%sql
USE CATALOG setu_rail;
USE SCHEMA bronze;

## Step 1 — Ingest schedules.json (389K records — the core dataset)

In [0]:
# schedules.json is a flat JSON array — Spark reads it natively
VOLUME = "/Volumes/setu_rail/bronze/raw_files/kaggle_ds"

schedules_raw = (spark.read
    .option("multiLine", "true")
    .json(f"{VOLUME}/schedules.json")
)

print(f"Schedules records: {schedules_raw.count():,}")
schedules_raw.printSchema()
schedules_raw.show(5, truncate=False)

In [0]:
from pyspark.sql.functions import col, when

# Clean: arrival='None' string → proper null
schedules_clean = (schedules_raw
    .withColumn("arrival",
        when(col("arrival") == "None", None).otherwise(col("arrival")))
)

(schedules_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.bronze.schedules"))

print("✅ bronze.schedules written")

## Step 2 — Ingest trains.json (GeoJSON FeatureCollection → flatten to rows)

In [0]:
from pyspark.sql.functions import explode, col

trains_geo = (spark.read
    .option("multiLine", "true")
    .json(f"{VOLUME}/trains.json")
)

# Explode features array and flatten properties
trains_flat = (trains_geo
    .select(explode(col("features")).alias("f"))
    .select(
        col("f.properties.number").alias("train_no"),
        col("f.properties.name").alias("train_name"),
        col("f.properties.type").alias("train_type"),
        col("f.properties.zone").alias("zone"),
        col("f.properties.from_station_code").alias("from_code"),
        col("f.properties.to_station_code").alias("to_code"),
        col("f.properties.from_station_name").alias("from_name"),
        col("f.properties.to_station_name").alias("to_name"),
        col("f.properties.departure").alias("origin_departure"),
        col("f.properties.arrival").alias("terminus_arrival"),
        col("f.properties.distance").alias("total_distance_km"),
        col("f.properties.duration_h").alias("duration_hours"),
        col("f.properties.duration_m").alias("duration_minutes_extra"),
        col("f.properties.sleeper").alias("has_sleeper"),
        col("f.properties.third_ac").alias("has_3ac"),
        col("f.properties.second_ac").alias("has_2ac"),
        col("f.properties.first_ac").alias("has_1ac"),
        col("f.properties.chair_car").alias("has_chair"),
    )
)

print(f"Trains: {trains_flat.count():,}")
trains_flat.show(5, truncate=False)

(trains_flat.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.bronze.trains"))

print("✅ bronze.trains written")

## Step 3 — Ingest stations.json (GeoJSON → flatten with lat/lon)

In [0]:
from pyspark.sql.functions import col, get_json_object, size

stations_geo = (spark.read
    .option("multiLine", "true")
    .json(f"{VOLUME}/stations.json")
)

stations_flat = (stations_geo
    .select(explode(col("features")).alias("f"))
    .select(
        col("f.properties.code").alias("station_code"),
        col("f.properties.name").alias("station_name"),
        col("f.properties.state").alias("state"),
        col("f.properties.zone").alias("zone"),
        col("f.properties.address").alias("address"),
        col("f.geometry.coordinates")[0].alias("longitude"),
        col("f.geometry.coordinates")[1].alias("latitude"),
    )
    .filter(col("station_code").isNotNull())
)

print(f"Stations: {stations_flat.count():,}")
stations_flat.show(5, truncate=False)

(stations_flat.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.bronze.stations"))

print("✅ bronze.stations written")

## Step 4 — Ingest Air Quality CSV and pivot to wide format

In [0]:
# --- Air Quality ingestion (Step 4) — FIXED ---
from pyspark.sql.functions import upper, trim, when, avg as spark_avg, to_timestamp, to_date, col
VOLUME2 = "/Volumes/setu_rail/bronze/raw_files"
def find_file_recursive(root_path: str, target_name: str) -> str:
    """Recursively find a file by name in directory tree."""
    try:
        items = dbutils.fs.ls(root_path)
    except Exception as e:
        raise FileNotFoundError(f"Cannot list {root_path}: {e}")
    
    for item in items:
        if item.name == target_name:
            return item.path
    
    for item in items:
        if item.name.endswith("/"):
            try:
                result = find_file_recursive(item.path, target_name)
                if result:
                    return result
            except FileNotFoundError:
                continue
    
    raise FileNotFoundError(f"Could not find '{target_name}' in {root_path}")

# Find AQ CSV recursively
try:
    aq_path = find_file_recursive(VOLUME2, "india_air_quality.csv")
    print(f"✅ Found AQ file: {aq_path}")
except FileNotFoundError as e:
    print(f"❌ {e}")
    aq_path = None

if not aq_path:
    raise FileNotFoundError("india_air_quality.csv not found in Volume or subdirectories")

# Read with explicit schema (avoid inference errors)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("country", StringType(), True),
    StructField("state", StringType(), True),
    StructField("city", StringType(), True),
    StructField("station", StringType(), True),
    StructField("last_update", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("pollutant_id", StringType(), True),
    StructField("pollutant_min", DoubleType(), True),
    StructField("pollutant_max", DoubleType(), True),
    StructField("pollutant_avg", StringType(), True),
])

aq_raw = (spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .schema(schema)
    .csv(aq_path))

print(f"AQ rows: {aq_raw.count():,}")

# Clean and pivot
aq_clean = (aq_raw
    .withColumn("city_u", upper(trim(col("city"))))
    .withColumn("obs_ts", to_timestamp(col("last_update"), "dd-MM-yyyy HH:mm:ss"))
    .withColumn("obs_date", to_date(col("obs_ts")))
    .withColumn("pollutant_id", trim(col("pollutant_id")))
    .withColumn("pollutant_avg_num",
        when((col("pollutant_avg") == "NA") | (col("pollutant_avg").isNull()), None)
        .otherwise(col("pollutant_avg").cast("double")))
)

# Pivot: one row per city/date with pollutant columns
aq_wide = (aq_clean
    .groupBy("city_u", "obs_date", "latitude", "longitude")
    .pivot("pollutant_id", ["PM2.5", "PM10", "NO2", "SO2", "CO", "OZONE", "NH3"])
    .agg(spark_avg("pollutant_avg_num"))
    .withColumnRenamed("city_u", "city")
    .withColumnRenamed("PM2.5", "pm25")
    .withColumnRenamed("PM10", "pm10")
    .withColumnRenamed("NO2", "no2")
    .withColumnRenamed("SO2", "so2")
    .withColumnRenamed("CO", "co")
    .withColumnRenamed("OZONE", "ozone")
    .withColumnRenamed("NH3", "nh3")
)

print(f"AQ pivoted rows: {aq_wide.count():,}")
aq_wide.show(5, truncate=False)

(aq_wide.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.bronze.air_quality"))

print("✅ bronze.air_quality written")

## Step 5 — Wildcard ingest Southern Railway CSVs

In [0]:
VOLUME3 = "/Volumes/setu_rail/bronze/raw_files/data_gov"
import re
from pyspark.sql.functions import col

# Read all train_*.csv files with wildcard
sr_csv = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mergeSchema", "true")
    .option("mode", "PERMISSIVE")
    .csv(f"{VOLUME3}/train_*.csv")
)

print(f"Southern Railway CSV rows: {sr_csv.count()}")
print(f"Original columns ({len(sr_csv.columns)}): {sr_csv.columns[:10]}...")

# ── Sanitize column names with deduplication ──────────────────────
def sanitize_and_deduplicate(columns: list) -> list:
    """
    Sanitize column names AND handle duplicates by adding numeric suffix.
    """
    def sanitize(name: str) -> str:
        if not name:
            return "unknown"
        name = name.replace('\n', ' ').replace('\t', ' ')
        name = re.sub(r'[^\w\s]', '', name)
        name = re.sub(r'\s+', '_', name.strip())
        name = name.lower()
        return name if name else "unknown"
    
    sanitized = [sanitize(c) for c in columns]
    
    # Handle duplicates: if a name appears multiple times, add suffix
    seen = {}
    result = []
    for name in sanitized:
        if name in seen:
            seen[name] += 1
            result.append(f"{name}_{seen[name]}")
        else:
            seen[name] = 0
            result.append(name)
    
    return result

new_columns = sanitize_and_deduplicate(sr_csv.columns)

# Check for any remaining duplicates
if len(new_columns) != len(set(new_columns)):
    print("⚠️  Warning: Duplicate columns after sanitization:")
    from collections import Counter
    dups = [name for name, count in Counter(new_columns).items() if count > 1]
    print(f"  Duplicates: {dups}")
else:
    print(f"✅ All {len(new_columns)} columns are unique")

print(f"Sanitized columns: {new_columns[:10]}...")

sr_csv_clean = sr_csv.toDF(*new_columns)

sr_csv_clean.show(3, truncate=False)

(sr_csv_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.bronze.sr_timings"))

print("✅ bronze.sr_timings written")

## Step 6 — Delta depth: OPTIMIZE + DESCRIBE DETAIL

In [0]:
%sql
OPTIMIZE setu_rail.bronze.schedules;
DESCRIBE DETAIL setu_rail.bronze.schedules;

In [0]:
%sql
-- Time Travel — this is what judges see live in the demo
DESCRIBE HISTORY setu_rail.bronze.schedules;

In [0]:
%sql
-- Summary
SELECT 'schedules' AS tbl, COUNT(*) AS rows FROM setu_rail.bronze.schedules
UNION ALL SELECT 'trains',       COUNT(*) FROM setu_rail.bronze.trains
UNION ALL SELECT 'stations',     COUNT(*) FROM setu_rail.bronze.stations
UNION ALL SELECT 'air_quality',  COUNT(*) FROM setu_rail.bronze.air_quality
UNION ALL SELECT 'sr_timings',   COUNT(*) FROM setu_rail.bronze.sr_timings;

✅ **Next:** `02_build_silver_enriched.ipynb`